# Early Detection of Collective Psychological Pressure

> **Aggregate, population-level language signals only. No diagnosis. No individual-level analysis.**

This notebook detects stress-related *linguistic signals* in public online text and reports only aggregate patterns
(cells with n < K_MIN are suppressed). It does not assess, profile, rank or re-identify any individual.
Findings are phrased as "stress-related language in the sampled communities shifted", never as "the population became stressed".

**Runs on any Windows laptop: just Run All.**
- **0.0** installs what is missing and picks the torch build for this laptop. It trains on the laptop's NVIDIA GPU (CUDA) if
  it has one, otherwise on the CPU.
- **Nothing runs twice.** Every step that computes or writes something runs once. Later Run Alls print
  `SKIP <step>: already done on <machine> at <time>` and load the saved result. Training starts automatically for whatever is not
  trained yet, and an interrupted seed resumes. To redo a step on purpose, add its number to `FORCE_RERUN` in cell 0.1.
- **What ran where, and when:** `trained_models/HANDOFF.md` (one row per step, with machine, device, time and duration).
- **Raw datasets are optional.** Steps 1–2 need manually downloaded raw data (SAD, Mendeley, SenticNet, Zenodo). On a laptop
  without it they print `NOT RUN …`, and Step 3 still works: it only needs Dreaddit, which downloads from Hugging Face.

Cells marked `# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<` train models.

## Step 0: Setup, configuration and device

In [ ]:
# Step 0.0: SETUP for THIS Windows laptop (yours or anyone's). Installs the libraries into this kernel's Python and makes
# sure torch can use the laptop's NVIDIA GPU. Safe to re-run: pip runs only when something is missing, at the wrong
# version, or when torch cannot use the GPU (e.g. a CPU-only torch is installed on a laptop that has an NVIDIA GPU).
# Torch build per laptop (picked from nvidia-smi):
#   NVIDIA GPU    -> CUDA build: 12.1 (driver >= 528), 11.8 (older driver), 12.8 for RTX 50-series (needs torch 2.7.1,
#                    UNVERIFIED with the pinned transformers 4.46.3: tell the assistant if training errors there)
#   no NVIDIA GPU -> CPU build (training works, but takes hours per seed)
import importlib.metadata as md
import json, os, platform, re, subprocess, sys, tempfile
from pathlib import Path

if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f"Python {sys.version.split()[0]} at {sys.executable}: torch / pyspark need Python 3.10-3.12 "
                       "(3.11 recommended). In VS Code pick a 3.11 environment (Select Kernel > Python Environments).")

_root = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (c / "code" / "config.yaml").is_file())
_req = _root / "requirements.txt"
PT = "https://download.pytorch.org/whl/"


def _pins(req_file):
    """[(name, version)] for every `name==version` line (extras like psycopg[binary] are stripped)."""
    hits = (re.match(r"^\s*([A-Za-z0-9_.\-]+)(?:\[[^\]]*\])?==([^\s;#]+)", l) for l in req_file.read_text(encoding="utf-8").splitlines())
    return [m.groups() for m in hits if m]


def _missing_or_wrong(req_file, skip=("torch",)):
    """(name, wanted, installed) for every pin that is not satisfied (torch is handled separately, per laptop)."""
    bad = []
    for name, want in _pins(req_file):
        if name.lower() in skip:
            continue
        try:
            have = md.version(name)
        except md.PackageNotFoundError:
            have = None
        if have is None or have.split("+")[0] != want:
            bad.append((name, want, have))
    return bad


def _pip(*args):
    print("$ python -m pip", " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", *args], text=True, capture_output=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode:
        raise RuntimeError(f"pip failed (exit {r.returncode}); see the output above")


def _nvidia_gpu():
    """First NVIDIA GPU via nvidia-smi (works before torch is installed), or None."""
    for fields in ("name,compute_cap,memory.total,driver_version", "name,memory.total,driver_version"):
        try:
            r = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True, timeout=30)
        except (OSError, subprocess.SubprocessError):
            return None
        if r.returncode == 0 and r.stdout.strip():
            v = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
            if len(v) == 4:
                return {"name": v[0], "cap": float(v[1]), "vram_gb": round(float(v[2]) / 1024, 1), "driver": v[3]}
            return {"name": v[0], "cap": None, "vram_gb": round(float(v[1]) / 1024, 1), "driver": v[2]}
    return None


def _torch_plan(gpu):
    """(torch version, wheel index URL, needs a working CUDA GPU?, why) for this laptop."""
    if gpu is None:
        return "2.5.1", PT + "cpu", False, "no NVIDIA GPU found (nvidia-smi): CPU build"
    drv = float(".".join(gpu["driver"].split(".")[:2]))
    if gpu["cap"] is not None and gpu["cap"] < 5.0:
        return "2.5.1", PT + "cpu", False, f"{gpu['name']} (compute {gpu['cap']}) is too old for current torch: CPU build"
    if gpu["cap"] is not None and gpu["cap"] >= 10.0:
        if drv < 570:
            print(f"  ! driver {gpu['driver']} is too old for {gpu['name']}: update the NVIDIA driver to >= 570")
        return "2.7.1", PT + "cu128", True, f"{gpu['name']} (compute {gpu['cap']}, RTX 50 / Blackwell) needs CUDA 12.8 wheels"
    if drv >= 528.33:
        return "2.5.1", PT + "cu121", True, f"{gpu['name']} (compute {gpu['cap']}), driver {gpu['driver']}: CUDA 12.1 build"
    if drv < 452.39:
        print(f"  ! driver {gpu['driver']} is very old: update the NVIDIA driver, then re-run 0.0")
    return "2.5.1", PT + "cu118", True, f"{gpu['name']}, driver {gpu['driver']} (< CUDA 12 minimum): CUDA 11.8 build"


_PROBE = """
import json, torch
r = {"version": torch.__version__, "cuda": torch.version.cuda, "available": torch.cuda.is_available(), "works": False}
if r["available"]:
    try:
        x = torch.randn(256, 256, device="cuda"); float((x @ x).sum())     # a real kernel launch, not just a flag
        r.update(works=True, device=torch.cuda.get_device_name(0))
    except Exception as e:
        r["error"] = str(e)[:300]
print(json.dumps(r))
"""


def _probe_torch():
    r = subprocess.run([sys.executable, "-c", _PROBE], capture_output=True, text=True)   # fresh process: sees new wheels
    return json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else {"error": r.stderr[-300:]}


print(f"Python {sys.version.split()[0]} | {platform.platform()} | {sys.executable}")
GPU = _nvidia_gpu()
TORCH_VERSION, TORCH_INDEX, WANT_GPU, why = _torch_plan(GPU)
print("NVIDIA GPU:", f"{GPU['name']} | {GPU['vram_gb']} GB | compute {GPU['cap']} | driver {GPU['driver']}" if GPU else "none")
print(f"torch plan: {TORCH_VERSION} from {TORCH_INDEX} ({why})")
_imported_before = [m for m in ("torch", "transformers", "numpy", "pandas", "pyarrow", "sklearn") if m in sys.modules]

# 1) everything except torch (torch line removed so pip never swaps the right torch build for another one)
_todo = _missing_or_wrong(_req)
if _todo:
    print(f"{len(_todo)} package(s) to install/fix: " + ", ".join(f"{n} (have {h or 'none'}, want {w})" for n, w, h in _todo))
    _tmp = Path(tempfile.gettempdir()) / "stress_signals_requirements_no_torch.txt"
    _tmp.write_text("\n".join(l for l in _req.read_text(encoding="utf-8").splitlines()
                              if not re.match(r"^\s*torch==", l)), encoding="utf-8")
    _pip("install", "-r", str(_tmp))
else:
    print(f"all {len(_pins(_req)) - 1} other pinned packages already installed")

# 2) torch: right version AND, if this laptop has a supported NVIDIA GPU, that GPU really runs a computation
T = _probe_torch()
_ok = T.get("version", "").split("+")[0] == TORCH_VERSION and (not WANT_GPU or T.get("works"))
if not _ok:
    print(f"torch needs (re)installing: have {T.get('version', 'none')} (CUDA {T.get('cuda')}, GPU works {T.get('works')})"
          + (f" | error: {T['error']}" if T.get("error") else ""))
    _pip("install", "--force-reinstall", f"torch=={TORCH_VERSION}", "--index-url", TORCH_INDEX)
    T = _probe_torch()

# 3) the project package itself (editable, so code edits apply)
try:
    md.version("stress_signals")
except md.PackageNotFoundError:
    _pip("install", "-e", str(_root / "code"), "--no-deps")

print(f"\ntorch {T.get('version')} | CUDA build {T.get('cuda')} | GPU runs computations {T.get('works')}")
if T.get("works"):
    print(f"TRAINING DEVICE: {T['device']} (cuda, {GPU['vram_gb']} GB)")
elif WANT_GPU:
    print(f"!!! GPU FOUND BUT NOT USABLE BY TORCH: {T.get('error', 'not available')}. Update the GPU driver and re-run 0.0; "
          "until then training runs on the CPU (hours per seed).")
else:
    print("TRAINING DEVICE: CPU (no supported NVIDIA GPU on this laptop). Training works but takes hours per seed.")
if (_todo or not _ok) and _imported_before:
    print(f"\n>>> RESTART THE KERNEL now (already imported before the install: {_imported_before}), then Run All again. <<<")

In [ ]:
# Step 0.1: bootstrap (safe to re-run). Imports the package, loads + validates config, seeds, versions.
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "code" / "config.yaml").is_file():
            return c
    raise FileNotFoundError("code/config.yaml not found above " + str(start))

PROJECT_ROOT = _find_root(Path.cwd().resolve())   # folder holding code/ and data/; all config paths resolve against it
SRC = str(PROJECT_ROOT / "code")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from stress_signals import ETHICS_BANNER, __version__, get_logger, load_config, set_seed
from stress_signals import handoff as H
from stress_signals.utils import device_label, library_versions, models_root, raw_data_status

# Re-run steps on purpose (normally empty): e.g. {"3C.1", "3D.1"} or {"2.5"}. Training is never forced: to retrain a seed,
# move its folder trained_models/stress/runs/<model>/seed<k> somewhere else first.
FORCE_RERUN = set()

CFG = load_config(PROJECT_ROOT / "code" / "config.yaml")
LOG = get_logger("stress_signals", CFG["logging"]["level"], CFG["_paths"]["logs"] / "pipeline.log")
SEED = CFG["primary_seed"]
set_seed(SEED, deterministic=CFG["deterministic"])
from stress_signals import hf_sync

# Pull missing datasets + the trained stress model from the private Hugging Face repos (config.yaml: huggingface). Needs
# `hf auth login` once on this laptop; never overwrites local files; prints a note instead of failing if unreachable.
for _k, _note in hf_sync.pull_missing(CFG).items():
    print(f"hugging face [{_k}]: {_note}")
RAW = raw_data_status(CFG)
RAW_OK = all(RAW.values())
RAW_WHY = (f"raw datasets missing on this laptop: {[k for k, v in RAW.items() if not v]} (manual downloads, see README). "
           "Step 3 does not need them.")

print("=" * 88)
print(ETHICS_BANNER)
print("=" * 88)
print(f"stress_signals {__version__} | project root: {PROJECT_ROOT}")
print(f"config: {CFG['_config_file']} | seeds: {CFG['seeds']} (primary {SEED}) | K_MIN: {CFG['privacy']['k_min']}")
print(f"models + handoff log: {models_root(CFG)} | device: {device_label()}")
print(f"raw datasets present: {RAW}" + ("" if RAW_OK else "  -> Steps 1-2 limited to Dreaddit"))
if FORCE_RERUN:
    print("FORCE_RERUN:", sorted(FORCE_RERUN))
print("library versions:")
for name, ver in library_versions().items():
    print(f"  {name:<13} {ver if ver else 'NOT INSTALLED'}")

In [ ]:
# Step 0.2: environment check (read-only): the device training and inference will use on THIS laptop.
from stress_signals.utils import detect_environment

ENV = detect_environment()
for key in ("python", "platform", "torch", "torch_cuda_build", "cuda_available", "gpu_name",
            "gpu_capability", "vram_total_gb", "vram_free_gb", "java"):
    print(f"{key:<18} {ENV[key]}")
print(f"\nTRAINING DEVICE: {device_label()}")
for w in ENV["warnings"]:
    print("  !", w)

## Step 1: Acquire and audit all datasets

Read-only inspection: no model weights are created or updated. Outputs: `data/processed/_cache/audit_<name>.json|.md`,
`data/outputs/reports/data_audit.md`, `data/outputs/manifests/step1_audit_manifest.json`. No raw text is written to them.

**Kaggle credentials (GoEmotions Kaggle copy, optional):** kaggle.com → *Settings* → *API* → *Create New Token* downloads
`kaggle.json`. Save it as `%USERPROFILE%\.kaggle\kaggle.json`, or set the `KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables. Never commit it. Without credentials the
audit records the Kaggle copy as `unavailable` and uses the official files plus the HF fallback.

Needs the manually downloaded raw datasets (see README). Without them these cells print `NOT RUN …`.

In [ ]:
# Step 1.1: Zenodo header check (reads only the header row of teaching_post_features_tfidf_256.csv)
from stress_signals import audit as A
if H.ready(RAW_OK, "1.1", RAW_WHY):
    head = A.zenodo_header(CFG, "teaching", "post")
    print("columns:", head["n_columns"])
    print("non-feature columns:", head["non_feature_columns"])
    for k in ("text_column_candidates", "author_columns", "date_columns", "id_columns"):
        print(f"{k:<24}", head[k] or "NONE")

In [ ]:
# Step 1.2: run all audits ONCE (~2 min). Later Run Alls load the saved audit summaries instead.
import json

def _saved_audits():
    return {p.stem.removeprefix("audit_"): json.loads(p.read_text(encoding="utf-8"))
            for p in sorted(CFG["_paths"]["interim"].glob("audit_*.json")) if p.stem != "audit_senticnet_handcheck"}

if H.ready(RAW_OK, "1.2", RAW_WHY):
    results = H.run_step(CFG, "1.2/audits", "Audit all datasets", lambda: A.run(CFG),
                         done_marker=CFG["_paths"]["manifests"] / "step1_audit_manifest.json",
                         skip_result=_saved_audits, force="1.2" in FORCE_RERUN)
    for name, res in results.items():
        print(f"{name:<14}", "ERROR " + res["error"] if "error" in res else "ok", "| flags:", res.get("flags", []))
    print("report:", CFG["_paths"]["reports"] / "data_audit.md")

In [ ]:
# Step 1.3: SenticNet eyeball view: NOTEBOOK VIEW ONLY (scrubbed, truncated, 3 rows per file x label). Not saved anywhere.
import pandas as pd
from stress_signals.privacy import scrub_text
if H.ready(RAW_OK, "1.3", RAW_WHY):
    pd.set_option("display.max_colwidth", 220)
    for name, df in A.load_senticnet(CFG).items():
        tcol, lcol = A.senticnet_text_label_cols(df)
        view = df.groupby(lcol, group_keys=False)[list(df.columns)].apply(lambda g: g.sample(min(3, len(g)), random_state=SEED))
        view = view.assign(text=view[tcol].astype(str).map(lambda t: scrub_text(t, CFG)[:220]))[[lcol, "text"]]
        print(f"--- {name} (text column={tcol!r}, label column={lcol!r})")
        display(view)

In [ ]:
# Step 1.4: create the hand-check sheet (~50 rows stratified by file x label) if it does not exist (never overwritten).
# Open it, fill `human_label` with 0/1 using the definition "the text expresses stress" (blank if unsure), save it.
if H.ready(RAW_OK, "1.4", RAW_WHY):
    SHEET = A.make_senticnet_handcheck(CFG)
    print(SHEET)

In [ ]:
# Step 1.5: score the hand-check, then rebuild the report + manifest. Runs once per version of the sheet: it runs again
# only after you edit (fill in) the sheet.
from stress_signals.utils import sha256_file
if H.ready(RAW_OK, "1.5", RAW_WHY):
    def _score():
        out = A.score_senticnet_handcheck(CFG)
        A.audit_senticnet(CFG)
        A.build_report(CFG)
        A.write_audit_manifest(CFG)
        return out
    HC = H.run_step(CFG, f"1.5/handcheck/{sha256_file(SHEET)[:12]}", "Score the SenticNet hand-check", _score,
                    skip_result=lambda: json.loads((CFG["_paths"]["interim"] / "audit_senticnet_handcheck.json").read_text(encoding="utf-8")),
                    force="1.5" in FORCE_RERUN)
    print(HC)

## Step 2: Preprocessing, unified schema, splits

No model weights are created or updated here. Code: `code/stress_signals/preprocess.py`; tests: `code/tests/test_preprocess.py`.

**Unified schema** (first 8 columns of every `data/processed/*.parquet`): `record_id` (sha256 of source + native id),
`source`, `community`, `created_at` (UTC), `text_clean`, `n_chars`, `n_tokens_est` (regex word+punctuation count, *not* model
tokens), `lang`. Label/split columns follow. No author, username or URL fields exist anywhere.

**Decisions applied (approved in Step 1):**
- Dreaddit: exact-duplicate train texts dropped. If the copies disagree on the label, the whole group is dropped. Train texts that also appear in test are dropped. Official test = FINAL TEST.
  Train/validation comes from `StratifiedGroupKFold` on `post_id`. This is the stratified version of GroupShuffleSplit.
- GoEmotions: official files only, with the official train/dev/test splits (dev is renamed validation). Kaggle copy dropped.
- Zenodo pilot: only `subreddit, date, post` are read. `author` and the feature columns are never read. Text is deduplicated by hash across all files.
  The pre window really is 2018-11-01 to 2019-11-01. `created_at` is day-resolution only.
- SenticNet: validation-only parquet. Its labels are automated. Twitter_Non-Advert is deduplicated against Twitter_Full,
  rows that contain Dreaddit segments are removed, and duplicate groups with conflicting labels are dropped.
- SAD: cleaned into the unified schema. **No split here.** Step 5 owns the stressor splits and the Reddit gold set.

**Filters:** the language and minimum-length filters apply to unlabelled corpora only. Labelled benchmark rows, including official
test rows, are never dropped by them. `lang` is still recorded for them.
**Language ID: lingua-language-detector.** It is deterministic and offline (models ship in the wheel, which matters for Spark workers),
and its authors report better short-text accuracy than langdetect/langid/fastText (our check: UNVERIFIED).
**Chunking:** sentence-aligned windows of at most `max_length` tokens with a `stride`-token overlap. The bound is re-checked by re-tokenising.
**Aggregation:** the default is the *mean* of calibrated chunk probabilities. The max grows with the number of chunks, so long-post
communities (r/relationships) would look systematically more stress-laden than short-post ones. That would bias the aggregates this project reports.
The mean is length-neutral; `max` stays available for sensitivity analysis.

**On a laptop without the raw datasets** only Dreaddit is built (downloaded from Hugging Face). Its fixed split file is in git and is verified, never regenerated.

In [ ]:
# Step 2.1: cleaning demo on SYNTHETIC strings (no user text)
from collections import Counter
from stress_signals import preprocess as P

demo = [
    "OMG &amp;#x200B; **I'm SO stressed** about [rent](https://example.com/x) <url> 😭 ask u/someone or mail a.b@example.org",
    "> quoted\n\n# Heading\n* bullet\n[removed]",
    "Call (555) 123-4567 about the 2019 2020 2021 taxes...",
]
st = Counter()
for s in demo:
    c = P.clean_text(s, CFG, st)
    print(repr(c))
    print("   lexical:", P.clean_text_lexical(c, CFG, already_clean=True))
    assert P.clean_text(c, CFG) == c        # idempotent
print("edit counts:", dict(st))

In [ ]:
# Step 2.2: build the processed datasets + fixed splits + manifest ONCE. All datasets when the raw data is on this laptop,
# otherwise Dreaddit only (downloaded from Hugging Face). Zenodo takes ~10-20 min the first time (resumable cache).
# Saved splits are NEVER regenerated (P.run(CFG, force_splits=True) would).
_which = None if RAW_OK else ["dreaddit"]
_marker = (CFG["_paths"]["manifests"] / "step2_preprocess_manifest.json") if RAW_OK else (CFG["_paths"]["processed"] / "dreaddit.parquet")

def _saved_step2():
    m = json.loads((CFG["_paths"]["manifests"] / "step2_preprocess_manifest.json").read_text(encoding="utf-8"))
    return m.get("extra", {}).get("datasets", {})

RES2 = H.run_step(CFG, "2.2/processed/" + ("all" if RAW_OK else "dreaddit"), "Build processed datasets + splits",
                  lambda: P.run(CFG, only=_which, force="2.2" in FORCE_RERUN), done_marker=_marker,
                  skip_result=_saved_step2, force="2.2" in FORCE_RERUN)
for name, r in RES2.items():
    print(f"{name:<11} final_rows={r['final_rows']}")
print("report:", CFG["_paths"]["reports"] / "preprocess_report.md")

In [ ]:
# Step 2.3: split checks (aggregate counts only; datasets that are not on this laptop are left out)
dre = pd.read_parquet(CFG["_paths"]["processed"] / "dreaddit.parquet")
print("Dreaddit split x label:\n", pd.crosstab(dre["split"], dre["label"], margins=True))
print("post_id overlap between splits (must all be 0):", P.group_overlap(dre))
print("official test rows all in FINAL TEST:", set(dre.loc[dre.official_split == "test", "split"]) == {"test"})
if (CFG["_paths"]["processed"] / "goemotions.parquet").exists():
    goe = pd.read_parquet(CFG["_paths"]["processed"] / "goemotions.parquet", columns=["split", "labels"])
    print("GoEmotions split sizes:", goe["split"].value_counts().to_dict())
for name in ("sad", "zenodo_pilot", "senticnet_validation"):
    p = CFG["_paths"]["processed"] / f"{name}.parquet"
    if p.exists():
        df = pd.read_parquet(p, columns=["community", "lang", "n_tokens_est"])
        print(f"{name}: rows={len(df)}, lang={df['lang'].fillna('und').value_counts().head(3).to_dict()}, "
              f"median n_tokens_est={df['n_tokens_est'].median()}")

In [ ]:
# Step 2.4: chunking check on the Zenodo pilot ONCE: counts only, seeded sample, tokenizer of the stress model
def _chunking_check():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(CFG["models"]["stress"]["base_model"], use_fast=True)
    ML, STRIDE, MAXC = CFG["models"]["stress"]["max_length"], CFG["chunking"]["stride"], CFG["chunking"]["max_chunks_per_doc"]
    zp = pd.read_parquet(CFG["_paths"]["processed"] / "zenodo_pilot.parquet", columns=["community", "text_clean"])
    sample = pd.concat([g.sample(min(len(g), 300), random_state=SEED) for _, g in zp.groupby("community")])
    rows = []
    for comm, t in zip(sample["community"], sample["text_clean"]):
        ch = P.chunk_text(t, tok, ML, STRIDE, MAXC)
        assert all(c.n_tokens <= ML for c in ch)
        rows.append((comm, ch[0].n_chunks_total, max(c.n_tokens for c in ch)))
    cs = pd.DataFrame(rows, columns=["community", "n_chunks", "max_chunk_tokens"])
    print(f"window={ML}, overlap={STRIDE}, max chunks kept={MAXC}; max tokens seen in any chunk: {cs.max_chunk_tokens.max()}")
    print(cs.groupby("community")["n_chunks"].describe(percentiles=[.5, .9, .99]).round(2))
    print("share of records needing >1 chunk:", round(float((cs.n_chunks > 1).mean()), 3),
          "| share with > max_chunks (subsampled):", round(float((cs.n_chunks > MAXC).mean()), 4))
    print("aggregation example (mean of chunk probabilities):", P.aggregate_chunk_scores([0.2, 0.7, 0.6]))

if H.ready((CFG["_paths"]["processed"] / "zenodo_pilot.parquet").exists(), "2.4", RAW_WHY):
    H.run_step(CFG, "2.4/chunking_check", "Chunking check on the Zenodo pilot", _chunking_check, force="2.4" in FORCE_RERUN)

In [ ]:
# Step 2.5: unit tests (cleaning idempotence, no PII left, chunk bound, no post_id overlap, split protection, ...).
# Runs once per version of the code: it runs again only after code/stress_signals, code/tests or config.yaml changed.
# A failing test stops Run All here on purpose (fix it before training).
import os, subprocess
from stress_signals.utils import code_fingerprint

def _run_tests():
    r = subprocess.run([sys.executable, "-m", "pytest", str(PROJECT_ROOT / "code" / "tests"), "-q", "-p", "no:cacheprovider",
                        "--rootdir", str(PROJECT_ROOT)], cwd=str(PROJECT_ROOT / "code"), capture_output=True, text=True,
                       env={**os.environ, "PYTHONPATH": SRC, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout[-3000:], r.stderr[-2000:])
    if r.returncode:
        raise AssertionError("unit tests failed (see the output above)")

H.run_step(CFG, f"2.5/tests/{code_fingerprint(PROJECT_ROOT)}", "Unit tests for this code version", _run_tests,
           force="2.5" in FORCE_RERUN)

## Step 3: Stress-language classifier on Dreaddit

Code: `code/stress_signals/stress_model.py` (training, inference, shortcut checks, bundle), `code/stress_signals/metrics.py`
(metrics, bootstrap, calibration) and `code/stress_signals/handoff.py` (handoff log, run-once guard).

**Device.** Training and inference use the laptop's NVIDIA GPU (CUDA) if it has one, otherwise the CPU (cell 0.2 shows which).
Cell 3.0c sizes the batch to the GPU and keeps the effective batch at 16 everywhere, so results are comparable between laptops.
The GPU trains in fp16; the CPU in fp32.

**Run All.** The baseline (3A.1) and every seed (3B.1) train automatically if they are not done yet. A seed interrupted mid-training
resumes from its last epoch on the next Run All. Evaluation (3C–3F) waits until every seed exists. The optional leave-one-subreddit-out
trainings (3E.2) run only with `RUN_LOSO = True`. Everything is saved under `trained_models/`. Epoch checkpoints go to `data/outputs/checkpoints/`.

**Decisions.**
- Input is `text_clean` (the Step 2 cleaning of Dreaddit `text`) only. The LIWC/DAL/syntax/social columns are dropped explicitly.
- Model selection uses validation `f1_stress`. The test set is never used for selection, calibration or thresholds.
- `confidence` sample weights are off by default; the docstring of `confidence_weights` explains why.
- The deployable model is the seed with the best validation F1(stress). Temperature and threshold are fitted on validation.
  The threshold is searched only in 0.40-0.60, maximising validation F1(stress). The tuned threshold is **applied only if the validation gain over 0.5 is clear**:
  gain >= 0.01 and the 95% CI of the paired cluster-bootstrap gain is above 0. Otherwise 0.5 is used. Test metrics are reported at **both** 0.5 and the
  tuned threshold, once (guarded file).

In [ ]:
# Step 3.0a: settings, transformers argument names, data guard (no training)
from stress_signals import stress_model as SM, metrics as MX

MODEL_CHOICE = CFG["stress_model"]["model_choice"]     # "roberta-base" | "distilroberta-base" | "deberta-v3-base"
S3 = SM.settings(CFG, MODEL_CHOICE)
compat = SM.hf_compat()
print(f"model: {S3['model_choice']} -> {S3['hf_id']} | max_len {S3['max_length']} | lr {S3['learning_rate']} | "
      f"epochs {S3['epochs']} | warmup {S3['warmup_ratio']} | seeds {S3['seeds']}")
print(f"transformers {compat['transformers']}: eval key = {compat['eval_strategy_key']!r}, "
      f"Trainer tokenizer arg = {compat['tokenizer_key']!r}")
D3 = SM.load_dreaddit(CFG)                  # verifies the saved split file and keeps text-only columns
for k, v in D3.items():
    print(f"{k:<10} n={len(v):>5}  stress share={v['label'].mean():.3f}  columns={list(v.columns)}")
print("runs will be written to:", SM.runs_root(CFG))

In [ ]:
# Step 3.0b: LABEL MEANING CHECK. Notebook view only: scrubbed, truncated, 3 per label, never saved.
# The HF card only says `label: int64`. Confirm by reading: do label-1 texts express stress?
pd.set_option("display.max_colwidth", 200)
view = (D3["train"].groupby("label", group_keys=False)[list(D3["train"].columns)]
        .apply(lambda g: g.sample(3, random_state=SEED))[["label", "community", "text_clean"]])
view["text_clean"] = view["text_clean"].str.slice(0, 200)
display(view)

In [ ]:
# Step 3.0c: batch size for THIS laptop's device (session only; config.yaml is untouched). The EFFECTIVE batch stays 16
# (train_batch_size x gradient_accumulation_steps), so results are comparable between laptops; only memory per step changes.
import torch
from stress_signals.utils import select_device
DEVICE = select_device(CFG)
LOCAL_TRAIN_BATCH = "auto"        # "auto" | an int, e.g. 4 (if you hit "out of memory") | None: config.yaml settings
sm = CFG["stress_model"]
effective = sm["train_batch_size"] * sm["gradient_accumulation_steps"]
if LOCAL_TRAIN_BATCH == "auto":
    if DEVICE == "cuda":
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        LOCAL_TRAIN_BATCH = 16 if vram >= 14 else 8 if vram >= 7 else 4       # roberta-base, 256 tokens, fp16
    else:
        LOCAL_TRAIN_BATCH = None
if LOCAL_TRAIN_BATCH:
    LOCAL_TRAIN_BATCH = min(int(LOCAL_TRAIN_BATCH), effective)
    sm["train_batch_size"], sm["gradient_accumulation_steps"] = LOCAL_TRAIN_BATCH, max(1, effective // LOCAL_TRAIN_BATCH)
    sm["eval_batch_size"] = min(sm["eval_batch_size"], 2 * LOCAL_TRAIN_BATCH)
print(f"device {device_label(DEVICE)} | train batch {sm['train_batch_size']} x accumulation {sm['gradient_accumulation_steps']} "
      f"= effective {sm['train_batch_size'] * sm['gradient_accumulation_steps']} | eval batch {sm['eval_batch_size']} "
      f"| precision {'fp16' if DEVICE == 'cuda' else 'fp32'}")

### 3A. Baseline: TF-IDF (word 1-2 + char_wb 2-5) with logistic regression
This is a reference point. The transformer has to beat it by a clear margin, beyond the confidence intervals, to justify its cost. C is chosen on validation F1(stress).

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3A.1: fit the baseline ONCE (CPU by design, ~1-3 min). Skipped when it is already done.
BASE_DIR = SM.runs_root(CFG) / "baseline_tfidf_lr"
H.run_step(CFG, H.BASELINE_STEP, "Fit TF-IDF + logistic regression baseline", lambda: SM.fit_baseline(CFG, D3),
           done_marker=BASE_DIR / "DONE.json", skip_result=BASE_DIR)
if (BASE_DIR / "c_selection_validation.csv").exists():
    print(pd.read_csv(BASE_DIR / "c_selection_validation.csv").round(4))

In [ ]:
# Step 3A.2: baseline metrics ONCE (validation + test, threshold 0.5, cluster-bootstrap 95% CIs). Later: loaded from file.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["baseline"], "3A.2", R3["why_baseline"]):
    _f = BASE_DIR / "evaluation.json"
    BASE_EVAL = H.run_step(CFG, "3A.2/baseline_eval", "Evaluate the baseline", lambda: SM.evaluate_baseline(CFG),
                           done_marker=_f, skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")),
                           force="3A.2" in FORCE_RERUN)
    for split in ("validation", "test"):
        m, ci = BASE_EVAL[split]["all"], BASE_EVAL[split]["ci"]
        print(f"{split:<10} " + "  ".join(f"{k}={m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]"
                                          for k in ("f1_stress", "f1_macro", "roc_auc", "pr_auc", "accuracy")))

### 3B. Transformer fine-tuning
One run per seed, each resume-safe. Seeds that are done are skipped; an interrupted seed resumes from its last epoch checkpoint.
Speed depends on the laptop: a few minutes per epoch on a recent NVIDIA GPU, much longer on a small GPU, hours per seed on a CPU.

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3B.1: fine-tune every seed that is not done yet, on this laptop's best device.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if R3["seeds_missing"]:
    print(f"to train: seeds {R3['seeds_missing']} on {device_label(DEVICE)}")
    if DEVICE == "cpu":
        print("!!! NO GPU on this laptop: each seed takes HOURS on the CPU. Interrupt the kernel to stop; "
              "a re-run resumes from the last finished epoch.")
for seed in CFG["stress_model"]["seeds"]:
    rd = SM.run_dir_for(CFG, MODEL_CHOICE, seed)
    H.run_step(CFG, H.train_step_id(MODEL_CHOICE, seed), f"Fine-tune {MODEL_CHOICE}, seed {seed}",
               lambda seed=seed: SM.train_stress_seed(CFG, seed, MODEL_CHOICE), done_marker=rd / "DONE.json", skip_result=rd)
    if (rd / "train_summary.json").exists():
        summ = json.loads((rd / "train_summary.json").read_text())
        print(f"seed {seed}: best validation f1_stress = {summ.get('best_metric_validation_f1_stress')} "
              f"| epochs = {summ.get('epochs_completed')} | device = {summ.get('device')}")

In [ ]:
# Step 3B.2: training curves from the CSV logs (no training). Validation F1(stress) per epoch, per seed.
for seed, rd in SM.completed_seeds(CFG, MODEL_CHOICE).items():
    log = pd.read_csv(rd / "train_log.csv")
    ev = log[log["key"].isin(["eval_f1_stress", "eval_loss"])].pivot_table(index="epoch", columns="key", values="value")
    print(f"seed {seed}\n", ev.round(4))

### 3C. Evaluation (after training): validation and official test, uncalibrated at threshold 0.5
CIs come from a **cluster bootstrap over post_id** (1,000 resamples), because segments of the same post are not independent. *High-confidence* means
Dreaddit `confidence >= stress_model.high_confidence_threshold`. Reading these numbers: if the high-confidence F1 is much higher, many of the errors fall on
texts that annotators also disagreed about.

In [ ]:
# Step 3C.1: per-seed metrics + CIs, mean +/- std across seeds ONCE (reads saved logits). Later: loaded from file.
KEYS = ["accuracy", "precision_stress", "recall_stress", "f1_stress", "f1_macro", "roc_auc", "pr_auc"]

def _load_ev3(path):
    ev = json.loads(path.read_text(encoding="utf-8"))
    ev["seeds"] = {int(k): v for k, v in ev["seeds"].items()}        # JSON keys are strings
    return ev

R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.1", R3["why_seeds"]):
    _f = SM.runs_root(CFG) / MODEL_CHOICE / "evaluation_uncalibrated.json"
    EV3 = H.run_step(CFG, f"3C.1/evaluate_seeds/{MODEL_CHOICE}", "Evaluate all seeds (uncalibrated)",
                     lambda: SM.evaluate_runs(CFG, MODEL_CHOICE), done_marker=_f, skip_result=lambda: _load_ev3(_f),
                     force="3C.1" in FORCE_RERUN)
    for split in ("validation", "test"):
        rows = []
        for seed, r in EV3["seeds"].items():
            m, ci = r[split]["all"], r[split]["ci"]
            rows.append({"seed": seed, **{k: f"{m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]" for k in KEYS},
                         "confusion": m["confusion"]})
        print(f"\n== {split} (threshold 0.5, uncalibrated)"); display(pd.DataFrame(rows))
        print("mean +/- std across seeds:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3["summary"][split].items() if k in KEYS})

In [ ]:
# Step 3C.2: high-confidence subset (same metrics restricted to confident annotations; view of 3C.1)
if H.ready(R3["all_seeds"], "3C.2", R3["why_seeds"]):
    for split in ("validation", "test"):
        hc = {seed: r[split]["high_confidence"] for seed, r in EV3["seeds"].items()}
        first = next(iter(hc.values()))
        print(f"{split}: confidence >= {first['confidence_threshold']} keeps {first['share_of_split']:.1%} of rows (n={first['n']})")
        print("  mean +/- std:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3['summary'][f'{split}_high_confidence'].items() if k in KEYS})

In [ ]:
# Step 3C.3: final seed + per-subreddit error analysis (aggregates) + a small error view (scrubbed, truncated, never saved)
if H.ready(R3["all_seeds"], "3C.3", R3["why_seeds"]):
    FINAL_SEED = SM.select_final_seed(EV3)            # best VALIDATION f1_stress
    print("final seed (best validation f1_stress):", FINAL_SEED)
    display(SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE).round(3))
    te = SM.load_predictions(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED), "test").merge(
        D3["test"][["record_id", "text_clean"]], on="record_id")
    te["error"] = (te["p"] >= 0.5).astype(int) != te["label"]
    errs = te[te["error"]].assign(margin=lambda d: (d["p"] - 0.5).abs()).sort_values("margin", ascending=False)
    nshow = min(CFG["privacy"]["max_error_examples_in_notebook"], 10)
    view = errs.head(nshow)[["community", "label", "p", "confidence"]].assign(
        text=errs.head(nshow)["text_clean"].map(lambda t: scrub_text(t, CFG)[:180]))
    display(view)   # most confident mistakes: label noise, sarcasm, or topic shortcuts?

### 3D. Calibration and threshold (fitted on VALIDATION only), then the calibrated test report once
- **Temperature scaling** fits one scalar T that minimises validation NLL. It changes confidence but not the ranking, so ROC-AUC is unchanged.
- **Threshold:** search 0.40-0.60 for the best validation F1(stress). Apply it only if the gain over 0.5 is clear (gain >= `apply_rule.min_gain`
  and the paired-bootstrap CI is above 0). The gain is measured on the same validation set that chose the threshold, so it is optimistic; the minimum gain is the safety margin.
- **Reading ECE:** it is the average gap between predicted probability and observed frequency, so lower is better. Downstream, aggregate signals average calibrated probabilities,
  so calibration matters more for this project than the last point of F1.

In [ ]:
# Step 3D.1: fit T + threshold on validation ONCE; reliability diagram; ECE / Brier before vs after. Later: loaded from file.
if H.ready(R3["all_seeds"], "3D.1", R3["why_seeds"]):
    _f = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "calibration.json"
    CAL3 = H.run_step(CFG, f"3D.1/calibrate/{MODEL_CHOICE}/seed{FINAL_SEED}", "Calibrate the final seed on validation",
                      lambda: SM.calibrate(CFG, FINAL_SEED, MODEL_CHOICE), done_marker=_f,
                      skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")), force="3D.1" in FORCE_RERUN)
    dec = CAL3["threshold_decision"]; g = dec["validation_gain_f1_stress"]
    print(f"T = {CAL3['temperature']:.3f} | tuned threshold (searched {CAL3['threshold_search_grid'][:2]}) = {CAL3['threshold_tuned']:.2f}")
    print(f"validation F1(stress) gain over 0.5 = {g['gain']:+.4f} [95% CI {g['lo']:+.4f}, {g['hi']:+.4f}] "
          f"-> APPLIED threshold = {CAL3['threshold']:.2f} ({dec['reason']})")
    for k in ("before", "after"):
        v = CAL3[f"validation_{k}"]
        print(f"validation {k:<6}: ECE {v['ece']:.4f} | Brier {v['brier']:.4f} | F1(stress) {v['f1_stress']:.3f} @ {v['threshold']:.2f}")
    print(f"validation after, at tuned {CAL3['threshold_tuned']:.2f}: F1(stress) {CAL3['validation_after_tuned']['f1_stress']:.3f}")
    from IPython.display import Image
    display(Image(str(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "reliability_validation.png")))

In [ ]:
# Step 3D.2: FINAL calibrated test report: computed ONCE (FINAL_TEST_REPORT.json is reused afterwards).
# Reported at BOTH thresholds; "applied" is the one the decision rule picked on validation.
if H.ready(R3["all_seeds"], "3D.2", R3["why_seeds"]):
    _f = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "FINAL_TEST_REPORT.json"
    FIN3 = H.run_step(CFG, f"3D.2/final_test_report/{MODEL_CHOICE}/seed{FINAL_SEED}", "Final calibrated test report",
                      lambda: SM.final_test_report(CFG, FINAL_SEED, MODEL_CHOICE, force="3D.2" in FORCE_RERUN),
                      done_marker=_f, skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")),
                      force="3D.2" in FORCE_RERUN)
    print(f"TEST (T={FIN3['temperature']:.3f}); applied threshold = {FIN3['threshold']:.2f} [{FIN3['threshold_applied']}]: "
          f"{FIN3['threshold_decision_reason']}")
    rows = []
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        for k in KEYS + ["brier"]:
            rows.append({"threshold": f"{b['threshold']:.2f}", "metric": k, "value": round(b["test"][k], 3),
                         "ci95": f"[{b['test_ci'][k]['lo']:.3f}, {b['test_ci'][k]['hi']:.3f}]" if k in b["test_ci"] else ""})
    display(pd.DataFrame(rows).pivot(index="metric", columns="threshold", values=["value", "ci95"]))
    print("n =", FIN3["test"]["n"], "| ECE (threshold-free)", round(FIN3["test_ece"], 4))
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        print(f"  @{b['threshold']:.2f} confusion {b['test']['confusion']} | high-confidence F1(stress) "
              f"{b['test_high_confidence']['f1_stress']:.3f} (n={b['test_high_confidence']['n']})")

### 3E. Shortcut-learning checks
**(i) Topic masking (inference only).** Subreddit names and domain words (money, rent, abuse, boyfriend, ...) are replaced with the mask token.
A *control* condition masks the same number of random other words in each text, repeated over **5 random draws** (mean, sd and range are reported). How to read it:
- If the topic-masking drop is larger than **every** control draw's drop (`topic_drop_exceeds_all_control_draws`), the model leans on topic words, and its scores may reflect *what a
  community talks about* rather than how stressed the language is. That is a direct threat to comparisons between communities.
- If both drops are similar and small, the model mostly uses stress expression.
- Some drop is expected, because words like "anxiety" also carry real stress signal.

**(ii) Leave-one-subreddit-out (training, optional).** Train without one subreddit and test on all of its rows. How to read it: a large gap compared with in-domain
performance means the model does not generalise to unseen communities. That matters because the target corpus has communities Dreaddit lacks.

**(iii) Baseline top n-grams.** Look at the words with the largest weights. If the strongest stress words are topic words (`topic_term=True`) rather than
affect, coping or pressure words ("can't", "anxious", "scared", "overwhelmed"), the baseline is keying on topic.

In [ ]:
# Step 3E.1: topic masking vs random-masking control (5 draws) on the official test set ONCE (inference on this laptop's
# device). Uses the APPLIED threshold from 3D. Diagnostic only: nothing is tuned on these numbers. Later: loaded from file.
if H.ready(R3["all_seeds"], "3E.1", R3["why_seeds"]):
    model_dir = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "best_model"
    _f = SM.runs_root(CFG) / MODEL_CHOICE / "shortcut_masking.json"
    SHORT = H.run_step(CFG, f"3E.1/topic_masking/{MODEL_CHOICE}/seed{FINAL_SEED}", "Topic-masking shortcut check",
                       lambda: SM.shortcut_masking_eval(CFG, model_dir, "test", CAL3["temperature"], CAL3["threshold"], MODEL_CHOICE),
                       done_marker=_f, skip_result=lambda: pd.DataFrame(json.loads(_f.read_text(encoding="utf-8"))["results"]),
                       force="3E.1" in FORCE_RERUN)
    display(SHORT[["condition", "draw_seed", "f1_stress", "roc_auc", "accuracy", "predicted_positive_rate"]].round(3))
    print(f"texts with >=1 masked term: {SHORT['texts_with_masked_terms'].iloc[0]} of {SHORT['n'].iloc[0]}; "
          f"terms masked: {SHORT['masked_terms_total'].iloc[0]}")
    display(SM.summarise_masking(SHORT).round(4))

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3E.2 (OPTIONAL): leave-one-subreddit-out. Trains one model per held-out subreddit (distilroberta by default,
# 1 seed, config stress_model.shortcut.loso_*): ~10 extra trainings. Each subreddit runs ONCE; resume-safe like 3B.1.
RUN_LOSO = False      # True: train the LOSO models (Run All skips them while this is False)
LOSO_MODEL = CFG["stress_model"]["shortcut"]["loso_model"]
for sub in sorted(pd.concat(D3.values())["community"].unique()):
    d = SM.runs_root(CFG) / "loso" / f"{LOSO_MODEL}_{sub}"
    H.run_step(CFG, H.loso_step_id(LOSO_MODEL, sub), f"LOSO: train {LOSO_MODEL} without r/{sub}",
               lambda sub=sub: SM.train_loso(CFG, [sub])[sub], done_marker=d / "DONE.json", skip_result=d,
               allow=RUN_LOSO, blocked_reason="optional: set RUN_LOSO = True at the top of this cell")
LOSO = SM.evaluate_loso(CFG)
if not len(LOSO):
    print("no finished LOSO runs yet")
elif "FINAL_SEED" in globals():
    inorig = SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE)[["community", "n", "f1_stress", "roc_auc"]]
    display(LOSO.merge(inorig, left_on="heldout_subreddit", right_on="community", suffixes=("_heldout", "_in_domain_test")).round(3))
else:
    display(LOSO.round(3))

In [ ]:
# Step 3E.3: baseline n-grams with the largest weights (view; needs 3A.1). topic_term=True = contains a subreddit/domain term.
if H.ready(R3["baseline"], "3E.3", R3["why_baseline"]):
    NG = SM.top_ngrams(CFG)
    for direction in ("stress", "not_stress"):
        sub = NG[(NG["direction"] == direction) & (NG["type"] == "word")]
        print(f"\n{direction}: share of top word n-grams that are topic terms = {sub['topic_term'].mean():.2f}")
        display(sub[["rank", "ngram", "coef", "topic_term"]].head(20))

### 3F. Save the model bundle
This writes `trained_models/stress/v<YYYYMMDD>/` once. The bundle holds the weights and tokenizer,
`stress_config.json` (label map, max length, chunking rule, temperature, threshold, model name, seed), `metrics.json`, `model_card.md` and
`manifest.json` (SHA-256 of every file).

In [ ]:
# Step 3F: write the bundle ONCE (no training). Needs 3D.2; optionally 3A, 3E.1, 3E.2 so metrics.json includes them.
_ok = R3["all_seeds"] and "FINAL_SEED" in globals() and (SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "FINAL_TEST_REPORT.json").exists()
if H.ready(_ok, "3F", R3["why_seeds"]):
    def _bundle_from_log():
        ev = H.done_event(CFG, H.BUNDLE_STEP)
        return PROJECT_ROOT / ev["outputs"][0] if ev and ev["outputs"] else None
    BUNDLE3 = H.run_step(CFG, H.BUNDLE_STEP, f"Save stress model bundle ({MODEL_CHOICE}, seed {FINAL_SEED})",
                         lambda: SM.save_bundle(CFG, FINAL_SEED, MODEL_CHOICE, overwrite="3F" in FORCE_RERUN),
                         skip_result=_bundle_from_log, force="3F" in FORCE_RERUN)
    print("bundle:", BUNDLE3)
    if BUNDLE3 is not None and BUNDLE3.is_dir():
        for f in sorted(BUNDLE3.iterdir()):
            print(f"  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB")
        print((BUNDLE3 / "model_card.md").read_text()[:1500])

### Handoff summary
What ran, on which laptop and device, when, and how long it took (also in `trained_models/HANDOFF.md`).

In [ ]:
# Handoff summary: one row per step (state, machine/device, time, duration)
H.show(CFG)

## Step 4: Emotion model on GoEmotions

Code: `code/stress_signals/emotion_model.py` (data, training, calibration, thresholds, Ekman collapse, Dreaddit check, bundle)
and the multi-label helpers in `code/stress_signals/metrics.py`. Settings: `emotion_model:` in `code/config.yaml`.

> **Read before using any output of this step.**
> - **A negative emotion is not stress.** GoEmotions labels say which emotions raters read in a comment. Anger, sadness or
>   fear language is a separate signal from the stress classifier (Step 3) and is never relabelled as stress.
> - **Domain shift.** GoEmotions is short Reddit *comments* (median ~18 tokens). Dreaddit and the target corpora are longer
>   *posts*. 4F measures this; probabilities calibrated on GoEmotions are not guaranteed to be calibrated on posts.
> - Outputs describe language in sampled text, aggregated. Nothing here says anything about a person.

**Data (4A).** Official GoEmotions files (Step 1 found them identical to the HF `go_emotions/simplified` copy, 100% of ids,
texts and labels). The Kaggle copy was dropped in Step 1 (its contents were never verified), so `use_kaggle: false`. The loader reads
`data/processed/goemotions.parquet` and falls back to Hugging Face when that file is missing. Either way it checks every record id
against the fixed split file in git.

**Training (4B).** One sigmoid per label, `BCEWithLogits` loss. The model config stores `problem_type="multi_label_classification"`
(checked against the installed transformers by `tests/test_emotion_model.py`). Defaults: roberta-base, max length 128, lr 3e-5,
effective batch 32, 4 epochs, warmup 0.1, seeds 42/13/2024. Epochs and seeds are selected on validation macro average precision,
a threshold-free metric. Optional `class_weighting` (pos_weight) is off by default. Checkpoints go to `data/outputs/checkpoints/emotion/` every epoch,
and an interrupted seed resumes there.

**Where to train.** Training runs on this machine's NVIDIA GPU (fp16), otherwise on the CPU. For scale: each Step 3 stress seed
(2,398 segments × 4 epochs) took about 21 minutes on the Quadro P1000 (HANDOFF.md). GoEmotions has 43,410 training comments, about 18× as many,
but they are much shorter, so expect a seed to take considerably longer than a stress seed. **Faster GPU (optional):** copy the
project folder to a machine with a bigger NVIDIA GPU and Run All there. Without `data/processed/goemotions.parquet`, the data comes from the HF
fallback and is checked against the same split file. Then copy `trained_models/emotion/runs/` back; HANDOFF.md records where each seed ran.

**Order of evaluation.** 4C.1–4C.2 (all seeds, uncalibrated) → 4D (per-label calibration on validation) → 4C.3–4C.4 (per-label
thresholds on validation calibrated probabilities, then the test report, written once) → 4E Ekman → 4F Dreaddit → 4G bundle.
The test set is never used for any choice.

In [ ]:
# Step 4.0a (4A): settings, transformers argument names, GoEmotions data + 28-label multi-label format (no training)
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display
from stress_signals import emotion_model as EM, stress_model as SM
from stress_signals.privacy import scrub_text

EMO_CHOICE = CFG["emotion_model"]["model_choice"]        # "roberta-base" | "distilroberta-base"
S4 = EM.settings(CFG, EMO_CHOICE)
_compat = SM.hf_compat()
print(f"model: {S4['model_choice']} -> {S4['hf_id']} | problem_type {S4['problem_type']} | max_len {S4['max_length']} | "
      f"lr {S4['learning_rate']} | effective batch {S4['train_batch_size'] * S4['gradient_accumulation_steps']} | "
      f"epochs {S4['epochs']} | warmup {S4['warmup_ratio']} | seeds {S4['seeds']} | best-epoch metric {S4['metric_for_best_model']}")
print(f"transformers {_compat['transformers']}: eval key = {_compat['eval_strategy_key']!r}, "
      f"Trainer tokenizer arg = {_compat['tokenizer_key']!r}")

SOURCE4 = EM.resolve_source(CFG)
D4 = EM.load_goemotions(CFG, SOURCE4)           # verifies the fixed split file; validates every label row
NAMES4 = EM.label_names(CFG)
Y4 = {k: EM.label_matrix(v["labels"], len(NAMES4)) for k, v in D4.items()}
print(f"source: {SOURCE4} | {len(NAMES4)} labels: {', '.join(NAMES4)}")
for k, Y in Y4.items():
    print(f"{k:<10} matrix {Y.shape} dtype {Y.dtype} | rows with 0 labels: {int((Y.sum(1) == 0).sum())} | "
          f"label ids in range: {Y.shape[1] == 28}")
display(EM.structure_summary(D4, NAMES4))
display(EM.label_summary(D4, NAMES4))

In [ ]:
# Step 4.0b: LABEL FORMAT CHECK. Notebook view only: scrubbed, truncated, 1 comment for 6 labels, never saved.
pd.set_option("display.max_colwidth", 160)
_rows = []
for lab in ("anger", "fear", "nervousness", "sadness", "joy", "neutral"):
    j = NAMES4.index(lab)
    r = D4["train"][Y4["train"][:, j] == 1].sample(1, random_state=SEED).iloc[0]
    _rows.append({"label": lab, "all labels of this row": [NAMES4[i] for i in r["labels"]],
                  "text": scrub_text(r["text_clean"], CFG)[:160]})
display(pd.DataFrame(_rows))

In [ ]:
# Step 4.0c: batch size for THIS machine (session only; config.yaml is untouched). The EFFECTIVE batch stays 32
# (train_batch_size x gradient_accumulation_steps). Also measures how the installed Trainer treats gradient accumulation.
import torch
from stress_signals.utils import select_device
DEVICE4 = select_device(CFG)
LOCAL_TRAIN_BATCH_4 = "auto"      # "auto" | an int, e.g. 8 (if you hit "out of memory") | None: config.yaml settings
em = CFG["emotion_model"]
_eff = em["train_batch_size"] * em["gradient_accumulation_steps"]
if LOCAL_TRAIN_BATCH_4 == "auto":
    if DEVICE4 == "cuda":
        _vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        LOCAL_TRAIN_BATCH_4 = 32 if _vram >= 14 else 16 if _vram >= 7 else 8      # roberta-base, <=128 tokens, fp16
    else:
        LOCAL_TRAIN_BATCH_4 = None
if LOCAL_TRAIN_BATCH_4:
    LOCAL_TRAIN_BATCH_4 = min(int(LOCAL_TRAIN_BATCH_4), _eff)
    em["train_batch_size"], em["gradient_accumulation_steps"] = LOCAL_TRAIN_BATCH_4, max(1, _eff // LOCAL_TRAIN_BATCH_4)
    em["eval_batch_size"] = min(em["eval_batch_size"], 4 * LOCAL_TRAIN_BATCH_4)
GA_DIVIDED_BY_TRAINER = EM.trainer_divides_loss_by_ga()     # tiny random model, CPU, one backward pass, no optimizer step
print(f"device {device_label(DEVICE4)} | train batch {em['train_batch_size']} x accumulation {em['gradient_accumulation_steps']} "
      f"= effective {em['train_batch_size'] * em['gradient_accumulation_steps']} | eval batch {em['eval_batch_size']} "
      f"| precision {'fp16' if DEVICE4 == 'cuda' else 'fp32'}")
print(f"installed Trainer divides the loss by gradient_accumulation_steps: {GA_DIVIDED_BY_TRAINER} -> "
      f"{'nothing to correct' if GA_DIVIDED_BY_TRAINER else 'the emotion loss divides itself (EmotionTrainer), so one effective batch = one averaged gradient'}")

### 4B. Multi-label fine-tuning
One run per seed, each resume-safe. Seeds that are done are skipped, and an interrupted seed resumes from its last epoch checkpoint.
`RUN_EMOTION_TRAINING = False` in the cell below skips training on this Run All. Use it, for example, while the seeds train on another machine.

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 4B.1: fine-tune every emotion seed that is not done yet, on this machine's best device.
RUN_EMOTION_TRAINING = True
R4 = EM.readiness(CFG, EMO_CHOICE)
if R4["seeds_missing"] and RUN_EMOTION_TRAINING:
    print(f"to train: emotion seeds {R4['seeds_missing']} on {device_label(DEVICE4)}")
    if DEVICE4 == "cpu":
        print("!!! NO GPU on this machine: each seed takes MANY HOURS on the CPU. Interrupt the kernel to stop; "
              "a re-run resumes from the last finished epoch.")
for seed in CFG["emotion_model"]["seeds"]:
    rd = EM.run_dir_for(CFG, EMO_CHOICE, seed)
    H.run_step(CFG, H.emotion_train_step_id(EMO_CHOICE, seed), f"Fine-tune emotion {EMO_CHOICE}, seed {seed}",
               lambda seed=seed: EM.train_emotion_seed(CFG, seed, EMO_CHOICE, D4), done_marker=rd / "DONE.json",
               skip_result=rd, allow=RUN_EMOTION_TRAINING, blocked_reason="RUN_EMOTION_TRAINING = False")
    if (rd / "train_summary.json").exists():
        summ = json.loads((rd / "train_summary.json").read_text())
        print(f"seed {seed}: best validation macro AP = {summ.get('best_metric_validation_macro_ap')} | epochs = "
              f"{summ.get('epochs_completed')} | device = {summ.get('device')} | GA = {summ.get('gradient_accumulation')}")

In [ ]:
# Step 4B.2: training curves from the CSV logs (no training). Validation macro AP / macro-F1@0.5 / loss per epoch, per seed.
for seed, rd in EM.completed_seeds(CFG, EMO_CHOICE).items():
    log = pd.read_csv(rd / "train_log.csv")
    ev = log[log["key"].isin(["eval_macro_ap", "eval_f1_macro", "eval_loss"])].pivot_table(index="epoch", columns="key", values="value")
    print(f"seed {seed}\n", ev.round(4))

### 4C. Evaluation (after training)
4C.1 scores every seed on validation and test, **uncalibrated, at a global 0.5** (macro/micro F1, per-label P/R/F1, ROC-AUC and AP),
with mean ± std across seeds. 4C.2 picks the deployable seed by **validation** macro AP. Per-label thresholds and the
final test report with bootstrap CIs (4C.3–4C.4) come after calibration (4D), because thresholds are tuned on the
probabilities that are actually deployed.

In [ ]:
# Step 4C.1: per-seed metrics (uncalibrated, 0.5) + mean +/- std across seeds ONCE (reads saved logits). Later: loaded from file.
KEYS4 = ["f1_macro", "f1_micro", "precision_macro", "recall_macro", "roc_auc_macro", "ap_macro"]

def _load_ev4(path):
    ev = json.loads(path.read_text(encoding="utf-8"))
    ev["seeds"] = {int(k): v for k, v in ev["seeds"].items()}
    return ev

R4 = EM.readiness(CFG, EMO_CHOICE)
if H.ready(R4["all_seeds"], "4C.1", R4["why_seeds"]):
    _f = EM.runs_root(CFG) / EMO_CHOICE / "evaluation_uncalibrated.json"
    EV4 = H.run_step(CFG, f"4C.1/emotion_evaluate_seeds/{EMO_CHOICE}", "Evaluate all emotion seeds (uncalibrated)",
                     lambda: EM.evaluate_runs(CFG, EMO_CHOICE), done_marker=_f, skip_result=lambda: _load_ev4(_f),
                     force="4C.1" in FORCE_RERUN)
    for split in ("validation", "test"):
        display(pd.DataFrame([{"seed": s, **{k: round(r[split][k], 4) for k in KEYS4}} for s, r in EV4["seeds"].items()])
                .set_index("seed").rename_axis(f"{split} @0.5"))
        print("mean +/- std:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV4["summary"][split].items() if k in KEYS4})

In [ ]:
# Step 4C.2: final seed (best VALIDATION macro AP) + its per-label validation table (uncalibrated, 0.5)
if H.ready(R4["all_seeds"], "4C.2", R4["why_seeds"]):
    FINAL_SEED4 = EM.select_final_seed(EV4)
    print("final emotion seed (best validation macro AP):", FINAL_SEED4)
    display(EM.per_label_table(EV4["seeds"][FINAL_SEED4]["validation"])
            [["label", "support", "precision", "recall", "f1", "roc_auc", "average_precision"]].round(3))

### 4D. Per-label calibration (fitted on VALIDATION)
**Method: Platt scaling per label**, `p = sigmoid(a·logit + b)` with `a > 0`, fitted on validation logits only.
Dashboard aggregates average *probabilities* over many texts. A label whose probabilities are systematically too low or too high
(common for rare labels trained with BCE) therefore biases the aggregate. A per-label temperature only rescales the logit and keeps
`p = 0.5` at `logit = 0`, so it cannot remove such an offset; Platt's intercept `b` can. Because `a > 0`, ranking, ROC-AUC and AP stay
unchanged. Platt's smoothed targets keep the two-parameter fit stable for labels with few validation positives (grief has 13).
Set `calibration.method: temperature` to switch. Validation ECE is optimistic because it is measured on the data the fit used; the test ECE in 4C.4 is the honest check.

In [ ]:
# Step 4D.1: fit per-label calibration on validation ONCE; ECE before/after (per label + pooled); reliability diagram.
if H.ready(R4["all_seeds"], "4D.1", R4["why_seeds"]):
    _d4 = EM.run_dir_for(CFG, EMO_CHOICE, FINAL_SEED4)
    _f = _d4 / "calibration.json"
    CAL4 = H.run_step(CFG, f"4D.1/emotion_calibrate/{EMO_CHOICE}/seed{FINAL_SEED4}", "Calibrate the emotion model per label",
                      lambda: EM.calibrate(CFG, FINAL_SEED4, EMO_CHOICE), done_marker=_f,
                      skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")), force="4D.1" in FORCE_RERUN)
    b, a = CAL4["validation_ece_before"], CAL4["validation_ece_after"]
    print(f"method {CAL4['method']} | validation ECE pooled {b['pooled']:.4f} -> {a['pooled']:.4f} | "
          f"mean over labels {b['mean_over_labels']:.4f} -> {a['mean_over_labels']:.4f}")
    display(pd.DataFrame(CAL4["per_label"])[["label", "a", "b", "n_pos_validation", "ece_before", "ece_after",
                                             "mean_p_before", "mean_p_after", "observed_rate"]].round(4))
    display(Image(str(_d4 / "reliability_validation.png")))

### 4C (continued). Per-label thresholds on VALIDATION, then the final test report (once)
Each label gets the threshold that maximises **its** F1 on validation calibrated probabilities. The grid is 0.05–0.95; ties go to the
value closest to 0.5. Labels with fewer than 30 validation positives are flagged `low_support`, because their threshold is noisy.
Low-support labels keep 0.5 (user decision; `thresholds.low_support_rule: keep_0.5`), and their tuned value is still reported. The test report is written **once**.
It contains per-label P/R/F1/ROC-AUC/AP, macro and micro F1 with **95% bootstrap CIs** (1,000 row resamples), the same probabilities at
a global 0.5 for comparison, test ECE before and after calibration, and a sensitivity check without the test rows whose text
also occurs in train or validation (Step 2 flags).

In [ ]:
# Step 4C.3: per-label thresholds on VALIDATION calibrated probabilities ONCE. Later: loaded from file.
if H.ready(R4["all_seeds"], "4C.3", R4["why_seeds"]):
    _f = EM.run_dir_for(CFG, EMO_CHOICE, FINAL_SEED4) / "thresholds.json"
    THR4 = H.run_step(CFG, f"4C.3/emotion_thresholds/{EMO_CHOICE}/seed{FINAL_SEED4}", "Tune per-label emotion thresholds",
                      lambda: EM.tune_thresholds(CFG, FINAL_SEED4, EMO_CHOICE), done_marker=_f,
                      skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")), force="4C.3" in FORCE_RERUN)
    display(pd.DataFrame(THR4["per_label"]).round(3))
    v0, v1 = THR4["validation_at_0.5"], THR4["validation_at_applied"]
    print(f"validation macro-F1: global 0.5 {v0['f1_macro']:.3f} -> per-label {v1['f1_macro']:.3f} | micro-F1 "
          f"{v0['f1_micro']:.3f} -> {v1['f1_micro']:.3f}  ({THR4['caveat']})")

In [ ]:
# Step 4C.4: FINAL calibrated test report with per-label thresholds: computed ONCE (FINAL_TEST_REPORT.json is reused).
if H.ready(R4["all_seeds"], "4C.4", R4["why_seeds"]):
    _f = EM.run_dir_for(CFG, EMO_CHOICE, FINAL_SEED4) / "FINAL_TEST_REPORT.json"
    FIN4 = H.run_step(CFG, f"4C.4/emotion_final_test_report/{EMO_CHOICE}/seed{FINAL_SEED4}", "Final emotion test report",
                      lambda: EM.final_test_report(CFG, FINAL_SEED4, EMO_CHOICE, force="4C.4" in FORCE_RERUN),
                      done_marker=_f, skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")),
                      force="4C.4" in FORCE_RERUN)
    t, ci = FIN4["test"], FIN4["test_ci"]
    rows = [{"metric": k, "per-label thresholds": round(t[k], 3),
             "95% CI": f"[{ci[k]['lo']:.3f}, {ci[k]['hi']:.3f}]" if k in ci else "",
             "calibrated @0.5": round(FIN4["test_calibrated_at_0.5"][k], 3),
             "uncalibrated @0.5": round(FIN4["test_uncalibrated_at_0.5"][k], 3)} for k in KEYS4 + ["roc_auc_micro"]]
    display(pd.DataFrame(rows).set_index("metric"))
    display(EM.per_label_table(t, ci)[["label", "support", "threshold", "precision", "recall", "f1", "f1_lo", "f1_hi",
                                       "roc_auc", "roc_auc_lo", "roc_auc_hi", "average_precision"]].round(3))
    e = FIN4["test_ece"]
    print(f"test ECE pooled: uncalibrated {e['uncalibrated']['pooled']:.4f} -> calibrated {e['calibrated']['pooled']:.4f} | "
          f"mean over labels {e['uncalibrated']['mean_over_labels']:.4f} -> {e['calibrated']['mean_over_labels']:.4f}")
    s = FIN4.get("test_sensitivity_without_cross_split_overlap")
    if s:
        print(f"sensitivity: without {s['n_excluded']} test rows whose text also occurs in train/validation -> macro-F1 "
              f"{s['f1_macro']:.3f}, micro-F1 {s['f1_micro']:.3f} (n={s['n']})")

### 4E. Ekman collapse (dashboard readability)
The 27 emotions are collapsed into the 6 Ekman groups plus neutral, using the **official** `ekman_mapping.json` from the GoEmotions repo
(Step 1 audit: it covers all 27 emotions, and each emotion belongs to exactly one group).
**Probability rule (`ekman.combine: max`):** a group's probability is the **max** of its members' calibrated probabilities.
P(any member) ≥ max over members holds for any dependence between members, so `max` is a conservative lower bound that does not
grow with group size. Joy has 12 members, and members co-occur strongly (anger and annoyance appear together in 269 train rows), which is exactly when
`noisy_or` (1 − ∏(1 − p), which assumes independence) overcounts. `noisy_or` remains available for sensitivity runs.
**Decision rule:** a group is predicted when any member is above its own threshold. Its gold label is set when any member is labelled.
The group probability is a rule, not a separately calibrated model; the report shows its test ECE.

In [ ]:
# Step 4E.1: Ekman mapping, combination rule and test results (view of 4C.4; no new computation)
if H.ready(R4["all_seeds"] and "FIN4" in globals(), "4E.1", R4["why_seeds"]):
    EKMAN4 = EM.load_ekman_mapping(CFG, NAMES4)
    display(pd.DataFrame([{"group": g, "members": ", ".join(m), "n_members": len(m)} for g, m in EKMAN4.items()]))
    print("rule:", FIN4["ekman"]["rule"])
    display(pd.DataFrame(FIN4["ekman"]["test"]["per_group"]).assign(
        group_ece=lambda d: d["group"].map(FIN4["ekman"]["test_group_ece"])).round(3))
    print(f"Ekman test macro-F1 {FIN4['ekman']['test']['f1_macro']:.3f} | micro-F1 {FIN4['ekman']['test']['f1_micro']:.3f} | "
          f"macro ROC-AUC {FIN4['ekman']['test']['roc_auc_macro']:.3f}")

### 4F. Domain-shift check on Dreaddit (inference only)

> ⚠️ **GoEmotions is short Reddit comments; Dreaddit is longer posts.** The model was trained and calibrated on comments
> (median ~18 tokens). Dreaddit segments have a median of ~99 tokens and are split into sentence-aligned chunks of ≤ 128 tokens (mean of
> calibrated chunk probabilities). Its probabilities on posts are **not** guaranteed to be calibrated.
>
> ⚠️ **A negative emotion is not the same as stress.** This check asks only whether emotion language *differs* between
> segments that Dreaddit annotators labelled stress and not-stress. It does not validate the emotion model as a stress
> detector, and it says nothing about any writer.

What is computed: for the stress and not-stress segments, the mean calibrated probability and the share above threshold for each
emotion and Ekman group, plus the difference with a **95% CI from resampling whole posts** (segments of one post are not independent).
Per-community cells with n < K_MIN are suppressed. Output files: `domain_shift_dreaddit.json` (aggregates) and
`predictions_dreaddit.parquet` (ids + probabilities, no text; local only).
**Hand-check (4F.2):** a 200-row local CSV (100 stress and 100 not-stress segments, seeded; scrubbed text, no author and no post id) in
`data/processed/_cache/` (gitignored). Open it, fill the three `hand_*` columns, and report back the counts.

In [ ]:
# Step 4F.1: apply the final emotion model to Dreaddit text ONCE (inference on this machine's device). Later: loaded from file.
if H.ready(R4["all_seeds"] and "FIN4" in globals(), "4F.1", R4["why_seeds"]):
    print("!" * 100 + "\n" + EM.DOMAIN_SHIFT_NOTE + "\n" + "!" * 100)
    _d4 = EM.run_dir_for(CFG, EMO_CHOICE, FINAL_SEED4)
    _f = _d4 / "domain_shift_dreaddit.json"
    SHIFT4 = H.run_step(CFG, f"4F.1/emotion_domain_shift/{EMO_CHOICE}/seed{FINAL_SEED4}", "Emotion domain-shift check on Dreaddit",
                        lambda: EM.domain_shift_dreaddit(CFG, FINAL_SEED4, EMO_CHOICE), done_marker=_f,
                        skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")), force="4F.1" in FORCE_RERUN)
    print(f"segments: {SHIFT4['n']} | chunking: {SHIFT4['chunking']} | CI: {SHIFT4['ci_resampling']}")
    _cols = ["name", "mean_p_stress", "mean_p_not_stress", "diff_mean_p", "diff_lo", "diff_hi", "rate_stress", "rate_not_stress"]
    print("\nEkman groups (stress vs not-stress segments):")
    display(pd.DataFrame(SHIFT4["ekman"])[_cols].round(3))
    print("\n28 labels, sorted by difference in mean calibrated probability:")
    display(pd.DataFrame(SHIFT4["labels"])[_cols].sort_values("diff_mean_p", ascending=False).round(3))
    print(f"\nper community x Dreaddit label (Ekman mean probabilities; cells with n < {SHIFT4['k_min']} suppressed):")
    display(pd.DataFrame(SHIFT4["by_community_ekman"]).round(3))
    _png = _d4 / "domain_shift_dreaddit_ekman.png"
    EM.plot_domain_shift(SHIFT4, path=str(_png))
    display(Image(str(_png)))

In [ ]:
# Step 4F.2: 200-row LOCAL hand-check sheet ONCE (never overwritten): scrubbed text, no author / post id. Not displayed here.
if H.ready(R4["all_seeds"] and "SHIFT4" in globals(), "4F.2", R4["why_seeds"]):
    HANDCHECK4 = CFG["_paths"]["interim"] / f"emotion_dreaddit_handcheck_{EMO_CHOICE}_seed{FINAL_SEED4}.csv"
    H.run_step(CFG, f"4F.2/emotion_handcheck_sheet/{EMO_CHOICE}/seed{FINAL_SEED4}", "Write the emotion hand-check sheet",
               lambda: EM.write_handcheck_sample(CFG, FINAL_SEED4, HANDCHECK4, EMO_CHOICE), done_marker=HANDCHECK4,
               skip_result=HANDCHECK4)
    _hc = pd.read_csv(HANDCHECK4, encoding="utf-8-sig")
    print(f"hand-check sheet: {HANDCHECK4}\nrows by Dreaddit label: {_hc['dreaddit_label'].value_counts().to_dict()}")
    _filled = _hc["hand_top_emotion_plausible (y/n/unsure)"].fillna("").astype(str).str.strip().str.lower()
    if (_filled != "").any():
        print("filled so far (counts only):", pd.crosstab(_hc["dreaddit_label"], _filled).to_dict())
    else:
        print("Open it in Excel, fill hand_top_emotion_plausible (y/n/unsure), hand_missing_emotion, hand_notes; then re-run this cell.")

### 4G. Save the emotion model bundle
This writes `trained_models/emotion/v<YYYYMMDD>/` once. The bundle holds the weights and tokenizer, plus:
- `emotion_config.json`: label map, problem type, max length, per-label calibration `a`/`b`, thresholds, Ekman rule and chunking rule.
- `thresholds.json` and `ekman_mapping.json`.
- `metrics.json`: the final test report, all seeds, and the Dreaddit check.
- `model_card.md` and `manifest.json` (SHA-256 of every file).

In [ ]:
# Step 4G: write the emotion bundle ONCE (no training). Needs 4C.4; includes 4F if it ran.
_ok = R4["all_seeds"] and "FINAL_SEED4" in globals() and (EM.run_dir_for(CFG, EMO_CHOICE, FINAL_SEED4) / "FINAL_TEST_REPORT.json").exists()
if H.ready(_ok, "4G", R4["why_seeds"]):
    def _bundle4_from_log():
        ev = H.done_event(CFG, H.EMOTION_BUNDLE_STEP)
        return PROJECT_ROOT / ev["outputs"][0] if ev and ev["outputs"] else None
    BUNDLE4 = H.run_step(CFG, H.EMOTION_BUNDLE_STEP, f"Save emotion model bundle ({EMO_CHOICE}, seed {FINAL_SEED4})",
                         lambda: EM.save_bundle(CFG, FINAL_SEED4, EMO_CHOICE, overwrite="4G" in FORCE_RERUN),
                         skip_result=_bundle4_from_log, force="4G" in FORCE_RERUN)
    print("bundle:", BUNDLE4)
    if BUNDLE4 is not None and BUNDLE4.is_dir():
        for f in sorted(BUNDLE4.iterdir()):
            print(f"  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB")
        print((BUNDLE4 / "model_card.md").read_text(encoding="utf-8")[:2500])

In [ ]:
# Handoff summary after Step 4: one row per step (state, machine/device, time, duration)
H.show(CFG)

## Step 5: Stressor taxonomy and stressor classifier

Code: `code/stress_signals/stressor_model.py`. Taxonomy: `code/taxonomy/stressors_v1.yaml`. Settings: `stressor_model:` in
`code/config.yaml`. Annotation guideline: `data/outputs/reports/stressor_annotation_guideline.md`.

> **Read before using any output of this step.**
> - **Stressor labels are a research taxonomy of stressor LANGUAGE, not a clinical assessment.** They say what a text names
>   as a cause of pressure, aggregated over many texts. Nothing here says anything about a person.
> - **SAD is not Reddit.** It is short crowd-written / LiveJournal sentences. Every method is therefore judged on a
>   hand-labelled Reddit gold set (5B), and the SAD-test → gold-test drop is reported as the domain-shift cost.
> - `uncertainty` and `societal_problems` have no SAD labels: they are scored by zero-shot embedding similarity only and
>   are flagged **lower confidence** everywhere.

**Order.** 5A taxonomy + SAD split → 5B gold sample + annotation (**you annotate; everything that uses gold waits for it**)
→ 5C methods: (1) keywords, (3) zero-shot (no fitting), (2a) embeddings + logistic regression and (2b) distilroberta
fine-tune (both fit weights: banner cells) → SAD-test reports → gold scoring, thresholds on gold-dev, method choice on
gold-dev, gold-test report once → weak Dreaddit-subreddit sanity check → 5D bundle.

**Inference unit.** SAD is sentence-level, so a post is split into sentence windows (sentences under 4 words are merged
into the next one; at most 32 windows). Each window is scored, and the post score per category is the **max over windows**
(a stressor named in one sentence is named in the post; mean / top-2 mean are reported on dev as sensitivity). Supervised
methods use a **softmax over SAD's 9 categories** (relationship family/social as two classes summed into
`relationship_issues`; `emotional_turmoil` kept as a class but never output), temperature-calibrated on SAD-validation.
Multi-label output = every category whose post score reaches its threshold (tuned per category on gold-dev). No category
above threshold → `none_unclear`.

**Leakage.** Gold texts come only from Dreaddit's train and validation splits, so the Step 3 official-test evaluation is
untouched. Gold labels never train or tune the stress model. The stressor models never see Dreaddit text in training.
Gold-dev tunes thresholds and picks the method. Gold-test is scored once.

In [ ]:
# Step 5.0a (5A): taxonomy: validate, show the mapping tables, write the taxonomy report ONCE (after editing the YAML,
# add "5A.1" to FORCE_RERUN in cell 0.1). Aggregate tables only, no text.
import json
import numpy as np
import pandas as pd
from IPython.display import display
from stress_signals import stressor_model as STR

S5 = STR.settings(CFG)
TAX5 = STR.load_taxonomy(CFG)                      # raises with a list of problems if the YAML is inconsistent
CATS5 = STR.category_ids(TAX5)
print(STR.SCOPE_NOTE)
print(f"taxonomy v{TAX5['version']}: {len(CATS5)} stressor categories | classifier classes: {STR.classifier_classes(TAX5)}")
_sad_counts = pd.read_parquet(CFG["_paths"]["processed"] / "sad.parquet", columns=["top_label"])["top_label"].value_counts().to_dict()
display(STR.category_table(TAX5))
display(STR.sad_mapping_table(TAX5, _sad_counts))
display(STR.mendeley_table(TAX5))
for k, v in TAX5["decisions"].items():
    print(f"- {k}: {v}\n")
_rep5 = CFG["_paths"]["reports"] / f"stressor_taxonomy_v{TAX5['version']}.md"
H.run_step(CFG, "5A.1/taxonomy_report", "Write the stressor taxonomy report",
           lambda: STR.write_taxonomy_report(CFG, TAX5, _sad_counts), done_marker=_rep5, skip_result=_rep5,
           force="5A.1" in FORCE_RERUN)
print("report:", _rep5)

In [ ]:
# Step 5.0b: fixed SAD train/validation/test split (stratified by class), created ONCE and saved to
# data/processed/splits/sad_split_v1.json. Class counts per split (no text).
from stress_signals.preprocess import split_path
_sp5 = split_path(CFG, "sad")
H.run_step(CFG, "5A.2/sad_split", "Create the fixed SAD split", lambda: STR.load_sad_splits(CFG, TAX5) and _sp5,
           done_marker=_sp5, skip_result=_sp5)
SAD5 = STR.load_sad_splits(CFG, TAX5)              # verifies the saved split against the data
print({k: len(v) for k, v in SAD5.items()})
display(STR.class_counts(SAD5, TAX5))

In [ ]:
# Step 5.0c: embedding models: load both candidates and encode 3 SYNTHETIC sentences ONCE (inference only).
# Pooling / normalisation / max length are read from each checkpoint's own sentence-transformers files.
_f = STR.runs_root(CFG) / "embedding_check.json"
def _check5():
    from stress_signals.utils import write_json
    t = STR.verify_embedding_models(CFG)
    write_json(_f, {"rows": t.to_dict(orient="records")})
    return _f
H.run_step(CFG, "5C.0/embedding_check", "Verify the embedding models load", _check5, done_marker=_f, skip_result=_f)
display(pd.DataFrame(json.loads(_f.read_text(encoding="utf-8"))["rows"]))
EMB5 = STR.Embedder.from_cfg(CFG)                  # the configured model (stressor_model.embedding.model_choice)
print(f"using {EMB5.spec.hf_id} @ {EMB5.spec.revision} | pooling {EMB5.spec.pooling} | normalize {EMB5.spec.normalize} | "
      f"max_length {EMB5.spec.max_length} | lowercase {EMB5.spec.lowercase} | device {device_label(EMB5.device)}")

### 5B. Reddit gold set (you annotate)

5B.1 samples the gold set once: about 400 Dreaddit segments with **human stress label 1** and confidence of at least 0.6, from
the train and validation splits only. It takes one segment per post, stratified by subreddit (at least 20 per subreddit
where available, the rest proportional), split 50/50 into dev and test stratified by subreddit. The key file `data/processed/splits/stressor_gold_split_v1.json` holds ids
and subreddits only (no text, no Reddit post ids) and is versioned in git. The annotation sheets hold text and stay local
under `data/outputs/gold/`. **The subreddit is not shown in the sheets**, so it cannot bias the labels.

**Your task:** fill `stressor_gold_v1_annotator1.csv` following the guideline (`1` in every column that applies).
**Single annotator, no inter-annotator agreement** (user decision): there is no second sheet and no agreement statistic is
computed or simulated. This is stated in the gold-set report (`data/outputs/reports/stressor_gold_set_v1.md`), the
gold-test report, the model card and the limitations. The Zenodo top-up (~100 posts, `source = zenodo`) is added after Step 8.

In [ ]:
# Step 5B.1: sample the gold set + write the annotation sheets ONCE (never overwritten). Not displayed here.
_g5 = STR.gold_files(CFG)
GOLD5 = H.run_step(CFG, "5B.1/gold_sample", "Sample the stressor gold set + annotation sheets",
                   lambda: STR.build_gold_set(CFG, TAX5), done_marker=_g5["split"],
                   skip_result=lambda: STR.build_gold_set(CFG, TAX5))   # re-creates a deleted sheet from the key
print("key (ids only, in git):", _g5["split"])
print("annotator 1 sheet (LOCAL):", _g5["sheet1"])
print("gold-set report (counts only, in git):", _g5["report"])
print("annotation:", GOLD5["annotation"])
print("counts:", GOLD5["counts"])
print("\nleakage:", GOLD5["leakage"])

In [ ]:
# Step 5B.2: annotation progress (re-runs every time; counts only, no text) + refresh the gold-set report.
# Single annotator, no inter-annotator agreement (user decision): no agreement statistic is computed.
ST5 = STR.gold_status(CFG, TAX5)
for _k in ("sheet1",):
    _s = ST5[_k]
    if not _s["exists"]:
        print(f"{_k}: missing")
        continue
    print(f"{_k}: {_s['annotated']}/{_s['rows']} rows annotated (by split {_s['annotated_by_split']}) | problems: "
          f"{_s['n_problems']} {_s['problems_first_10']}")
    print("   positives so far:", _s["positives"])
GOLD_READY5 = ST5["complete"]
GOLD_WHY5 = ("the gold set is not fully annotated yet: fill every row of the annotator 1 sheet (see 5B), fix the listed "
             "problems, then Run All again")
print("annotation:", ST5["annotation"], "| gold-set report:", STR.write_gold_report(CFG, TAX5))

### 5C. Methods

(1) Keyword baseline and (3) zero-shot similarity fit nothing. (2a) fits a logistic regression on frozen sentence
embeddings of SAD-train (C chosen on SAD-validation). (2b) fine-tunes distilroberta-base on SAD-train (3 seeds, best epoch
and final seed chosen on SAD-validation macro-F1, gradient accumulation fixed at 1). Both cells below fit weights and
train on this laptop's GPU (CPU otherwise). An interrupted fine-tune seed resumes from its last epoch on the next Run All.

In [ ]:
# Step 5C.1: keyword baseline on SAD-test (sentence level, no fitting): category macro-F1 over the SAD-covered categories.
KW_SAD5 = STR.keyword_sad_report(CFG, TAX5, SAD5)
print(f"keyword SAD-test category macro-F1 (SAD-covered) = {KW_SAD5['test_category_macro_f1_sad_covered']:.3f} | "
      f"sentences with no / several keyword categories: {KW_SAD5['share_no_or_multiple_hits']:.1%}  ({KW_SAD5['rule']})")
display(pd.Series(KW_SAD5["test_category_f1"], name="F1").round(3).to_frame())

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 5C.3 (2a): logistic regression on frozen sentence embeddings of SAD-train (fits weights; minutes on this laptop).
RUN_STRESSOR_LR = True
_d = STR.lr_run_dir(CFG)
H.run_step(CFG, H.STRESSOR_LR_STEP, f"Fit embedding + logistic regression ({S5['embedding']['model_choice']})",
           lambda: STR.fit_embedding_lr(CFG, TAX5, SAD5, EMB5), done_marker=_d / "DONE.json", skip_result=_d,
           allow=RUN_STRESSOR_LR, blocked_reason="RUN_STRESSOR_LR = False")
if (_d / "lr_summary.json").exists():
    _s = json.loads((_d / "lr_summary.json").read_text(encoding="utf-8"))
    print(f"best C = {_s['best_C']} | SAD-validation macro-F1 = {_s['best_validation_macro_f1']:.3f}")
    display(pd.DataFrame(_s["C_grid"]).round(4))

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 5C.4 (2b): fine-tune distilroberta-base on SAD, every seed not done yet (fits weights; this laptop's GPU, else CPU).
RUN_STRESSOR_FINETUNE = True
_ft = S5["finetune"]
print(f"model {_ft['hf_id']} | max_len {_ft['max_length']} | lr {_ft['learning_rate']} | batch {_ft['train_batch_size']} | "
      f"epochs {_ft['epochs']} | seeds {_ft['seeds']} | best-epoch metric {_ft['metric_for_best_model']}")
for seed in _ft["seeds"]:
    rd = STR.ft_run_dir(CFG, seed)
    H.run_step(CFG, H.stressor_train_step_id(_ft["model_choice"], seed), f"Fine-tune stressor {_ft['model_choice']}, seed {seed}",
               lambda seed=seed: STR.train_finetune_seed(CFG, TAX5, seed, SAD5), done_marker=rd / "DONE.json",
               skip_result=rd, allow=RUN_STRESSOR_FINETUNE, blocked_reason="RUN_STRESSOR_FINETUNE = False")
    if (rd / "train_summary.json").exists():
        _s = json.loads((rd / "train_summary.json").read_text(encoding="utf-8"))
        print(f"seed {seed}: best SAD-validation macro-F1 = {_s['best_metric_validation_macro_f1']} | epochs "
              f"{_s['epochs_completed']} | device {_s['device']}")
R5 = STR.ft_readiness(CFG)

In [ ]:
# Step 5C.5: SAD calibration (temperature on SAD-validation) + SAD-test report ONCE per supervised method; final fine-tune
# seed = best SAD-validation macro-F1. Reads saved logits only.
SAD_REP5 = {"keyword": KW_SAD5}
if H.ready((STR.lr_run_dir(CFG) / "DONE.json").exists(), "5C.5/embed_lr", "embedding LR not fitted yet (5C.3)"):
    _d = STR.lr_run_dir(CFG)
    SAD_REP5["embed_lr"] = H.run_step(CFG, f"5C.5/sad_report/{STR.method_key(CFG, 'embed_lr')}", "SAD calibration + test (embed_lr)",
                                      lambda: STR.calibrate_and_report_sad(CFG, TAX5, _d, "embed_lr"),
                                      done_marker=_d / "SAD_TEST_REPORT.json",
                                      skip_result=lambda: json.loads((_d / "SAD_TEST_REPORT.json").read_text(encoding="utf-8")))
R5 = STR.ft_readiness(CFG)
if H.ready(R5["all_seeds"], "5C.5/finetune", R5["why"]):
    SEL5 = STR.ft_select_seed(CFG, TAX5)
    display(pd.DataFrame(SEL5["per_seed"]).round(3))
    print("final fine-tune seed:", SEL5["final_seed"], "|", SEL5["rule"])
    _d = STR.ft_run_dir(CFG, SEL5["final_seed"])
    SAD_REP5["finetune"] = H.run_step(CFG, f"5C.5/sad_report/{STR.method_key(CFG, 'finetune')}/seed{SEL5['final_seed']}",
                                      "SAD calibration + test (finetune)",
                                      lambda: STR.calibrate_and_report_sad(CFG, TAX5, _d, "finetune"),
                                      done_marker=_d / "SAD_TEST_REPORT.json",
                                      skip_result=lambda: json.loads((_d / "SAD_TEST_REPORT.json").read_text(encoding="utf-8")))
_rows = [{"method": m, "SAD-test category macro-F1 (SAD-covered)": r.get("test_category_macro_f1_sad_covered"),
          "class macro-F1 (9)": r.get("test_macro_f1_classes"), "accuracy": r.get("test_accuracy"),
          "temperature": (r.get("calibration") or {}).get("temperature")} for m, r in SAD_REP5.items()]
display(pd.DataFrame(_rows).round(3))

### 5C (continued). Reddit gold set: thresholds on gold-dev, method choice, gold-test once

These cells wait until the annotator 1 sheet is complete. Each method's windows are scored once (inference only, ids and
scores saved, no text). Per-category thresholds are tuned on **gold-dev only**. The method is chosen by gold-dev macro-F1
over categories with gold positives (`stressor_model.method_selection`). The gold-test report covers **every** method and is
written once, after all four methods are scored. A test-set ranking is listed for reference but never used for choosing.

In [ ]:
# Step 5C.6: score the gold set with every method that is ready (inference only) + tune thresholds on gold-dev ONCE each.
METHODS_READY5 = {"keyword": True, "zero_shot": True, "embed_lr": "embed_lr" in SAD_REP5, "finetune": "finetune" in SAD_REP5}
THR5 = {}
if H.ready(GOLD_READY5, "5C.6", GOLD_WHY5):
    GOLD_LAB5 = STR.load_gold_labels(CFG, TAX5)
    for _m, _ok in METHODS_READY5.items():
        if not H.ready(_ok, f"5C.6/{_m}", "its SAD model is not fitted/reported yet (5C.3-5C.5)"):
            continue
        _d = STR.gold_run_dir(CFG, _m)
        _key = STR.method_key(CFG, _m)
        H.run_step(CFG, f"5C.6/score_gold/{_key}", f"Score the gold set ({_key})",
                   lambda _m=_m: STR.score_gold(CFG, TAX5, _m, STR.build_scorer(CFG, TAX5, _m, EMB5), EMB5),
                   done_marker=_d / "window_scores.parquet", skip_result=_d / "window_scores.parquet")
        THR5[_m] = H.run_step(CFG, f"5C.6/tune_dev/{_key}", f"Tune thresholds on gold-dev ({_key})",
                              lambda _m=_m: STR.tune_on_dev(CFG, TAX5, _m, GOLD_LAB5), done_marker=_d / "thresholds.json",
                              skip_result=lambda _d=_d: json.loads((_d / "thresholds.json").read_text(encoding="utf-8")),
                              force="5C.6" in FORCE_RERUN)
    _rows = []
    for _m, t in THR5.items():
        dv = t["dev"]
        _rows.append({"method": _m, "dev macro-F1 (supported)": dv["f1_macro_supported"], "SAD-covered": dv["f1_macro_sad_covered"],
                      "zero-shot only": dv["f1_macro_zero_shot_only"], "micro-F1": dv["f1_micro"],
                      "none_unclear F1": dv["none_unclear"]["f1"], "labels/post": dv["mean_labels_per_post"],
                      **{f"agg {a}": v["f1_macro_supported"] for a, v in t["dev_aggregation_sensitivity"].items()}})
    print("gold-DEV (optimistic: thresholds tuned on these rows):")
    display(pd.DataFrame(_rows).round(3))
    for _m, t in THR5.items():
        print(f"\n{_m}: thresholds per category (dev support for EVERY category; global cosine threshold = "
              f"{t['global_cosine_threshold']}, used for cosine categories with < "
              f"{S5['thresholds']['low_support_dev_positives']} dev positives)")
        display(pd.DataFrame(t["per_category"])[["category", "kind", "n_pos_dev", "low_support", "rule", "tuned", "applied",
                                                  "f1_dev_at_tuned"]].round(3))

In [ ]:
# Step 5C.7: FINAL gold-test report (all four methods, dev-tuned thresholds, bootstrap CIs, domain-shift cost) ONCE.
_all5 = GOLD_READY5 and all(m in THR5 for m in STR.METHODS)
if H.ready(_all5, "5C.7", GOLD_WHY5 if not GOLD_READY5 else f"methods not scored yet: {[m for m in STR.METHODS if m not in THR5]}"):
    _f = STR.runs_root(CFG) / "gold" / "GOLD_TEST_REPORT.json"
    FIN5 = H.run_step(CFG, "5C.7/gold_test_report", "Final stressor gold-test report",
                      lambda: STR.final_gold_report(CFG, TAX5, STR.METHODS, GOLD_LAB5, SAD_REP5, force="5C.7" in FORCE_RERUN),
                      done_marker=_f, skip_result=lambda: json.loads(_f.read_text(encoding="utf-8")),
                      force="5C.7" in FORCE_RERUN)
    _rows = []
    for _m, r in FIN5["methods"].items():
        t, ci, ds = r["test"], r["test_ci_supported_categories"], r["domain_shift_cost"] or {}
        _rows.append({"method": _m, "dev macro-F1": r["dev"]["f1_macro_supported"], "TEST macro-F1": t["f1_macro_supported"],
                      "95% CI": f"[{ci['f1_macro']['lo']:.3f}, {ci['f1_macro']['hi']:.3f}]", "micro-F1": t["f1_micro"],
                      "SAD-covered": t["f1_macro_sad_covered"], "zero-shot only": t["f1_macro_zero_shot_only"],
                      "none_unclear F1": t["none_unclear"]["f1"], "SAD-test": ds.get("sad_test"), "drop": ds.get("drop")})
    display(pd.DataFrame(_rows).set_index("method").round(3))
    print(f"selected: {FIN5['selected_method']} ({FIN5['selection_rule']}) | test ranking for reference only: "
          f"{FIN5['test_ranking_for_reference']}")
    _sel = FIN5["methods"][FIN5["selected_method"]]
    _pl = pd.DataFrame(_sel["test"]["per_label"])[["label", "support", "threshold", "precision", "recall", "f1", "roc_auc"]]
    _pl["lower_confidence (zero-shot)"] = _pl["label"].map(lambda c: STR.coverage(TAX5)[c] == "zero_shot")
    _pl["support_dev"] = _pl["label"].map(lambda c: _sel["support"][c]["dev"])
    _pl["threshold_rule"] = _pl["label"].map(_sel["threshold_rule"])
    _pl["low_support_dev"] = _pl["label"].isin(_sel["low_support_categories"])
    display(_pl.rename(columns={"support": "support_test"}).round(3))
    print("support (gold positives) per category and split, every method uses the same gold labels:",
          {c: v for c, v in _sel["support"].items()})
    print(f"\ngold labels: {FIN5['annotation']}. {FIN5['annotation_limitation']}")

In [ ]:
# Step 5C.8: weak Dreaddit-subreddit cross-check for the selected method ONCE (inference; SANITY CHECK, NOT accuracy).
if H.ready("FIN5" in globals(), "5C.8", "needs the gold-test report (5C.7)"):
    _m = FIN5["selected_method"]
    _d = STR.gold_run_dir(CFG, _m)
    X5 = H.run_step(CFG, f"5C.8/crosscheck/{STR.method_key(CFG, _m)}", "Dreaddit subreddit cross-check",
                    lambda: STR.dreaddit_crosscheck(CFG, TAX5, _m, STR.build_scorer(CFG, TAX5, _m, EMB5),
                                                    THR5[_m]["thresholds"], EMB5),
                    done_marker=_d / "dreaddit_crosscheck.json",
                    skip_result=lambda: json.loads((_d / "dreaddit_crosscheck.json").read_text(encoding="utf-8")))
    print(X5["note"], f"| n = {X5['n_segments']} stress-positive segments not in the gold set")
    display(pd.DataFrame(X5["checks"]).round(3))
    print(f"per subreddit (cells with n < {X5['k_min']} suppressed):")
    display(pd.DataFrame(X5["by_subreddit"]).round(3))

### 5D. Save the stressor bundle

In [ ]:
# Step 5D: write the stressor bundle ONCE for the selected method (no training). Needs 5C.7 (5C.8 included if it ran).
if H.ready("FIN5" in globals(), "5D", "needs the gold-test report (5C.7)"):
    def _bundle5_from_log():
        ev = H.done_event(CFG, H.STRESSOR_BUNDLE_STEP)
        return PROJECT_ROOT / ev["outputs"][0] if ev and ev["outputs"] else None
    BUNDLE5 = H.run_step(CFG, H.STRESSOR_BUNDLE_STEP, f"Save stressor bundle ({FIN5['selected_method']})",
                         lambda: STR.save_bundle(CFG, TAX5, FIN5["selected_method"], overwrite="5D" in FORCE_RERUN),
                         skip_result=_bundle5_from_log, force="5D" in FORCE_RERUN)
    print("bundle:", BUNDLE5)
    if BUNDLE5 is not None and BUNDLE5.is_dir():
        for f in sorted(BUNDLE5.iterdir()):
            print(f"  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB")
        print((BUNDLE5 / "model_card.md").read_text(encoding="utf-8")[:3000])

In [ ]:
# Handoff summary after Step 5: one row per step (state, machine/device, time, duration)
H.show(CFG)

## Step 6: TensiStrength auxiliary signal and SenticNet external validation

_Not implemented yet. Filled in at Step 6._

## Step 7: Unified inference package and artifact bundle

_Not implemented yet. Filled in at Step 7._

## Step 8: Ingestion to a Parquet data lake (pilot, then main corpus)

_Not implemented yet. Filled in at Step 8._

## Step 9: PySpark distributed NLP inference

_Not implemented yet. Filled in at Step 9._

## Step 10: Spark analytics (temporal, emotions, stressors, topics)

_Not implemented yet. Filled in at Step 10._

## Step 11: Baseline and change detection -> collective pressure signals

_Not implemented yet. Filled in at Step 11._

## Step 12: PostgreSQL storage, aggregates only (optional)

_Not implemented yet. Filled in at Step 12._

## Step 13: Streamlit dashboard

_Not implemented yet. Filled in at Step 13._

## Step 14: Final evaluation report, ethics and documentation

_Not implemented yet. Filled in at Step 14._

## Step 15: Kafka + Spark Structured Streaming (optional)

_Not implemented yet. Filled in at Step 15._